<!-- dd:dd-lesson-es-1 -->

# Einsum notation

*Einsum · `es-1`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "es-1"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrlHWtv2zjyrwgBDohvbR0fEiUdsP9iv/mMheu43eLSJE0ct3uL/e8nzgwp6uFIpERvcwe0ik3rMUPOe4ajP25Elt38M/nj"
    "5uND/efm5fH+fLxZJzeH/cvxpR7Z/nHzcjy9Pv16eLw76jM+f3l6fD4lp8fnw2/J/iU5/ethn/ycnNLT8eHl8fl2u+XrROzW"
    "yVauk2y3W+Hd7u/1xbefXz4/vJz2D4fjLTzqdr9a19f+AtfWH80gXHX8/nQ8nI53v9YfnuHyX55fj/Wd9SOkfoSgR/y5TsLg"
    "pPtk6yRfJyoetBlBm9PEqGCoi0hAFn4Q3Z7S/fP+4dPxVmSr5G9JlfyUsFX6fHz5bf90vK0nVEUBk9U3rhdtnTA9k/XMFvXk"
    "rhNO81vCQhryo5Nphelk/S2HpdcnA9a7Gu8bUeZzGeE7LJWdl9SZEAFA10cxMi3fh6bl+9i01BOT1tjyFKYl059z/Fzqz1X9"
    "WX/ZCv1N4i9Kfy7wM8fL9fV44pbDqZzO5XAyp7MFnC24uSuHB3J6IodH8orOhfsICXeeTGKtmSzdiUSuvfZEXp48b6QsJ2+3"
    "lbk+CiKVP3QNV8savbvT70/Hn0/px/vH/UmKVfKPJGtRdA5MJWMuBEtFro8gOFla5LA2+iBgiKcwoscFERxcIVI4FChwceH0"
    "uKzvRBQu4SpY4ozulqUwoseJmHO4Kk/hgHeDpVcwrlIa0VcUSBIip1lHqVLwdy9VYIpKYm7iiK5MQfnA8CRJzDEoUeBMQWdy"
    "aeTKgDwBOSJoWXlupMr7kyZZe+7a0+ZOmD9S/zk+P77cajzo32pV/6TVJP3b1afVz4yDXT5HxIhsgojJriFiUKqAwBAkaEQj"
    "aFC20HiOogekDQkYK05AyKBMAXEhSPiIRsygZKHxHAUP8VOO97KSQzI2V3J8/vLpIuVPI/v6DkNTq4fHDDVSozRTsq1QSWAA"
    "yU+lHsSmoR81qqIk0E8eF0eroRpKYoZI2toJ6cvRTTAtjmZq6IcZ0rAaCZURQ7pyNBLNpNVH+tDSRvoIBz1aoigX8BEOerRC"
    "eS7gIxzweWQVEjcg8ROOnKS7wC94hF9QyiOmHFDliCvJesSWA7oc8SV5jxhzQJkDzp604RpXKE2jLX0A5Z7Sx4cjymqtt1fR"
    "YOOW9foHV77IpR1/4nZLqhKfF8MN1LffSEd7Mp8FaXuwXI0JklqKZBHxQJ7aABttGh1UgEhpn2Bt3u7Pjenb/tk30HD8vZab"
    "8XA1B8TwzYHQCAlz/NpN4dwtFk6c7LsNyjHmcpko3l94rb5zETr5QKYawiiQqVCwMAYEginanGlTIzSGxnQMjfPkp2STt8RO"
    "HgPYDQdNXi9VRTBTBIwtQ6uuntPHVRTe2wqjb4IFv5gS5ZhmRQbiQG4tWnIc7DeOVhsvbTgNfgPLTKAtJjg5zCT30aJEy0qA"
    "EBLoawhyNgTahwIdbbQFBViAooKIhQZGgTRTqDoUqBGFKkWZqBfaawpMNIWWqEL7U5noCFibCm1MBZalqtAipShOgVZjActX"
    "oG1YgDlYZBg98WPyWX594LJljvNe2tBHsMpyPOkY0OYGNmTzaqFAN7I58fgENg/y0jMTQAmMY6oJTqIgizwC+MjSOTBoWTNc"
    "WHhHdMI7vBXe0XZwBNAbGzs4wC3SPFaAW6Su4hJzKfpuf+rormwCReurhoCH8Yle2nShQTB6RtDkuO+yABqtaMeu4dt2DMMX"
    "VZeYXO9SGqEwSltzUBOODxsMdpHmcaGEB1hOyIp3527oXPGP6W7Yu/+oXgcX78XhqGeyrP9noLUyR3JXS9oicpq/EZYukN7i"
    "wDffkcWzQ7aVsVIpKQfUY1L8FMMIVvRWNmP6jtkUX6TkNoafITUYDHMeD7y8FQHKqv91c9vaqbJxu9BO1eapiGedSmECbmEs"
    "mZUTWDJihl3yVIFfLkv8oBSNcMmY/hRue29kvOqSCmcc6VstVOFgYLfCJALk9ZwEFQ2obtVFBNh0KGh+RUO8cgZR+sJnqXGT"
    "pXkUStQ3buIabFlCFI0s/L8iRAFz+sPSofQFzyVDFosMWZNXnR2MOPezxdNU/nkI9rNXvni6QXX2VKUNGuN0EYaISCUWVUiM"
    "QZT4vYLvXnh1CqtEE3w7U20Vb+JvaszACcOGmRizJ+QfX+/v3YowDTjUeESZ86pt684OLZ/5BWOg/km0fpITxfO5noGzGEQO"
    "f3lbK/PpknoI9s0bGAhbHsUgmhkTjWAstOrug74RsSd+U3mC3MiiUVFUGoS8romJrU7yNRZ1EbEyp4md1pPwoe1Zjmdq18mH"
    "wQCQHn+bANn05exmaafU+REy3pfFwhddu8DAoWgtj5/9Eg7ypvRbIlMsZOCkia9cM1COm6gzaEo0ZpcoFi1ecMIYH4aGY6FU"
    "hnLJ9FxQl1GWyiKFI12vXerLK2/5HGbNmuxp3EVjvnzThGk2VL/S4/WtnBKfnMHsynVZymU9Z1Q4zGzIaeq/meNlxMgJ05PC"
    "ooG6KuptRsgjRedZWuLOAEmF0lTIExz+a3R9nJlW4VUB3NlfZy1k+yUabTAnUowKI1fX81WalEhNQSy26bztP8/u2sxMkZeP"
    "X9k2r8fzWH3zetQYE5FnZKjOWJlKa/jK299Kr5gB77qmf4nPVOsT5b2wHQJmg7Dnpqg6Mvy5U7zNBqqpy2Via12WNWojVnTN"
    "1YHC7ql0d/8pu6cyMP42zpTRglQXC/kVbo3i9KcsOvzW7J6SRZf5aBRiI5pRPaNgXkwXhHrmH5rrM1o0imOXeSiPFV0YMGOb"
    "fX8Rbdnt1j5ImSKEzBbn7oLrw9UUx6jnT01xp0TUuZim63Z9tdhj1IY5aZerqx2b27XuLZ2buwxs9s+a3UosJfFQSbvzdcFg"
    "SR6d5KjmO2CDleiDy+ODawnCWUwyvFstC6rZCazXy3ZxzyzWo/spm9Rfa/thnQzWn5mfxgQjNXUQuY/ofm0XlI9GFVcBqhmn"
    "QG/wiz0HtMuwc8Tta/Qtk7Tb0P4l09ieYX4Q+If7qOc2bbAhsnCooohPFYUvMQxkePo4NEYlIsLiI+IamRthq/cdxlYxwjz9"
    "vEKUZg3c6aQhTUeSnS3MpN1SlekxEtyWYpG8QSCOHbtZdJSnVa9NqwHZ9lpzLB4zHSpUR80PX0uBF7xWUDMCYzBQjrLkwmlP"
    "UHbth4Hnvnmt0woC695MowLltQHs+5AlEGVlMm+KuhANjRsBvWT8D0ZAeGtbWE0Nc6XD08BGbDe4x8hSO1+IjY1NyVMtB4dm"
    "BcZHQsPM7tKbuohPrRTXjLjBDLC7z/GFPFttdVs8Bgkcvtt18Wi1YkEhGg+ZwjZ/KU30ww+jjrE84OrnEcHftLeczdalXy/G"
    "3r70xJrtR8am9tX4uk6+DCEK4xPSKD6c8tXf3P3imYJU4zp3BsaUf7H1Y242RuE3qCXzmpGWLTiwrMZS4xNqY2fgpry3cQ4Q"
    "Jh/EANz5DYsLfytno2Yn+Q9vxK7ugmNXh5p+h/CD8QmbWaabFoew2pg7v2KnbDxENQtlaollTUFkNPqK7Qk48/MrD63oyl0v"
    "GGTD7VbTxVzUSw/0waWtAFhPAfSw7KVWeWQsRdf/a/ubSly3jq1tUfJxY2ZGQA03z0mfrVK+ZW3ZUHR5QvwpGs6NA1miB4mR"
    "IuE5B70gbXf9RNyVY97wtuvcDJiFNWWrqAB3n2O4a6HOUpcD0sO85gracZzXyeES2vqnicnTboDpetmcyRr2sLiGnT99TXgl"
    "ryjDSq2GBPPbFDEt5yJbJGKHp2b/5iErshlN4xx1abAyfHHoiyxkyivgxEWrZSPny7L85ZD1Bx8mP1+WbecJqakL5k3cYvGA"
    "FM2HGBw+c+445mx5KjNpmkCW6MRmqjStWBS1jK3m8HzLVRoK4TGXdFqxGpNkjD8dcxpHDuMmB7HKroKOFWmUv5mt8b9d6CTQ"
    "7jEwscHAt3UyGKSG8YXbDHzzjBc5Ow27CZ4pGxUnlGjMwB47+glqAQ2d83zm4UJbsG+23UK/G8CkYvIZGDHfgC7iUT/+8O/b"
    "rUN32L5lnWx6Y7vVW31HIuIm3QAalQ6/vy5LnNru+G+xqPx284QAx6DdWvmDvROpCO4nPfxGpCzS1OFrWJRT4Xel/YPLo5P7"
    "7qS5Go1yEUqcaZzmX0WaB+4Um7LNJUYbMGpwQjS6UPd5q0LuPu8/3Q4VnQGknV+bveOrSO98WzBwMd5aAVsUR2xRz5sXV0i3"
    "wWPevHaiSE3TBfsyCXplBE/9vY7wliIzsHRfUlLMcCTo1VtRAG33W1BqYS5Cj7+eiuTv1v+PgojwtRj3nn19sXNXFHNFIok3"
    "Lbuh9XYwtRTRqKXwjnRfme+k7UfYtBBRf03SKCxHOCcPAbWKOQvPIE17b0kvor3I607C8Zb0HqGyaBraY7eE3RLdB/rIXacZ"
    "QeddeQt3fpBxaXGjXBasyuuy4NRcUhhyeXDjkXfJX0qli/cdiUt91Iog+F0/vLfVrYeBW/vKJ78SKLwxiePoVAs5OlfWzMw3"
    "QtvxWSb1UogVTmretZc1b8SrwqXyD2WivWmrr9ttTyK5t5y8Jaejwp//BUvFT44="
)
print("Delta Drills checker ready — 38 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-einsum-notation-model -->

## Reading an einsum spec string

`einsum.notation-model`


### anatomy of the spec string


`t.einsum` does one thing: it lets you **name the axes** of your tensors, then
describe the result by those names. Every call looks like

> `t.einsum('SPEC', tensor1, tensor2, ...)`

and the whole operation lives in that `'SPEC'` string. The string has exactly
two sides, split by `->`:

    'ik , kj -> ij'
     input1  input2   output

Read each slot:

- **Before `->` — the inputs.** One group per tensor you pass in, groups
  separated by a **comma**. Each group gives **one letter per axis**, in order:
  `ik` means "this tensor is 2-D; call axis 0 `i` and axis 1 `k`." The comma is
  literally "here comes the next tensor." Two groups → two tensors.
- **After `->` — the output.** The axes you want in the result, in the order
  you want them.
- **A letter is just a NAME for an axis** — like a loop variable. `i`, `x`, `q`
  carry no built-in meaning; only the PATTERN of where a letter appears matters.

Where each letter appears decides its fate — the three fates are the whole
language:

1. **Kept** — appears in the output → that axis survives (this lesson).
2. **Summed** — appears in an input but NOT the output → that axis is added up
   and disappears (next lesson).
3. **Multiplied** — the same letter shared across two inputs → those axes are
   paired and multiplied (a couple of lessons on).

Why name axes instead of using `permute`/`sum` with axis numbers?
Because the spec string IS the documentation: `'bchw->bhwc'` says "move channels
last" in plain axis names, where `transpose(x, (0,2,3,1))` makes you count.


> **Watch out.** - **Letters have no fixed meaning.** `'ij->ij'` and `'xy->xy'` are the same
  program; `i` is not "rows," `b` is not "batch" — those are conventions for
  humans reading your code, not rules PyTorch knows.
- **Always write the `->`.** Implicit mode (no arrow) exists but guesses the
  output by sorting letters alphabetically. In this course, always state the
  output side yourself.


The clearest way to see the grammar is the spec that does NOTHING: name every
axis, then ask for them back in the same order.


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3],
              [4, 5, 6]])

# 'ij->ij': input is 2-D (axis 0 = i, axis 1 = j); output lists i then j,
# the same order -> every axis kept, nothing reordered = the tensor unchanged.
same = t.einsum('ij->ij', a)
print(same)
# tensor([[1, 2, 3],
#         [4, 5, 6]])   <- exactly a, handed straight back




Why this is the anchor: `'ij->ij'` hands the tensor straight back — `print(same)`
shows the same numbers you started with. That proves the grammar in isolation:
the letters name the two axes, and the output side requests them unchanged.
Every other einsum spec is just a DEVIATION from this do-nothing baseline:
reorder the output letters (transpose), drop one (sum), or share one across two
inputs (multiply). Learn the baseline, and the rest are edits to it.

(The code is preloaded in the editor on the right — press Run and watch the
Output pane print the tensor — PyTorch wraps it in `tensor(...)`, which
is the surest sign you are looking at a tensor and not a list. That's the
fastest way to convince yourself.)


<!-- dd:dd-q244 -->

### Problem 244 · faded — your turn

Same tensor, but now emit the two axes in the OPPOSITE order — rows become
columns. You saw `'ij->ij'` return the tensor unchanged; change only the output
side so the layout flips (this is the transpose).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 3],
        [2, 4]])
```


In [ ]:
import torch as t

def solve(a):
    """Transpose via einsum: name the axes, then emit them in swapped order."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 2], [3, 4]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(244)


In [ ]:
#@title 💡 Solution — Problem 244
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('ij->ji', a)


print(solve(t.tensor([[1, 2], [3, 4]])))


### reordering the output = permutation


When every input letter also appears in the output — none dropped, none shared
— nothing is summed or multiplied. The ONLY thing the output side can do is set
the ORDER of the axes. That is a pure axis permutation.

- On a 2-D tensor, the only reorder is the transpose: `'ij->ji'`.
- On an N-D tensor, you can move any axis anywhere, and the spec documents the
  move in domain terms. `'bchw->bhwc'` takes a batch of channels-FIRST images
  `(batch, channel, height, width)` and lays them out channels-LAST
  `(batch, height, width, channel)` — the same numbers, re-indexed. The letters
  say exactly which axis went where; `x.permute(0, 2, 3, 1)` makes you
  decode the tuple.

No data changes — a permutation only relabels positions.


> **Watch out.** - **The output side is not decoration.** It decides which axes survive AND their
  order. Here every letter is kept, so it only reorders — but omit a letter and
  that axis would be SUMMED away (next lesson). Every letter you write, and every
  one you leave out, means something.


In [ ]:
import torch as t

imgs = t.arange(24).reshape(2, 3, 2, 2)   # (b=2, c=3, h=2, w=2), channels-first

# 'bchw->bhwc': keep all four axes, move channel to the end.
last = t.einsum('bchw->bhwc', imgs)
print(last.shape)          # (2, 2, 2, 3)  <- channels moved to the end

# Follow ONE element to see it only changed POSITION, not value:
print(imgs[0, 1, 1, 0])    # 5   (at b=0, c=1, h=1, w=0 in the original)
print(last[0, 1, 0, 1])    # 5   (same value, now at b=0, h=1, w=0, c=1)




Why: to check a relayout, print the shape and follow a single element — cheaper
and surer than eyeballing whole tensors. The shape shows channels landed last,
and the two prints both read `5`: the value didn't change, it only moved from
position `(b,c,h,w)=(0,1,1,0)` to `(b,h,w,c)=(0,1,0,1)`. That's what "permutation
relabels positions, never values" means, made concrete.


<!-- dd:dd-q285 -->

### Problem 285 · faded — your turn

A 4-D tensor `(b, h, s, d)`. This time don't move an axis to the end — SWAP the
two MIDDLE axes (`h` and `s`), leaving `b` first and `d` last. Same "reorder the
kept letters" idea as the channels-last move, a different target order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[1.],
          [1.]],

         [[1.],
          [1.]],

         [[1.],
          [1.]]]])
```


In [ ]:
import torch as t

def solve(x):
    """(b, h, s, d) -> (b, s, h, d): swap the two middle axes, ends fixed."""
    return t.einsum('_____', x)


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 3, 1))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(285)


In [ ]:
#@title 💡 Solution — Problem 285
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('bhsd->bshd', x)


print(solve(t.ones((1, 2, 3, 1))))


<!-- dd:dd-q271 -->

### Problem 271 · guided

Write a function solve(x) that takes a 4-D channels-second tensor of shape (b, c, h, w) and returns it in channels-LAST layout (b, h, w, c) — a pure axis permutation, no values changed.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 4.],
          [1., 5.]],

         [[2., 6.],
          [3., 7.]]]])
```


<details>
<summary>Hints</summary>

1. `(b, c, h, w) -> (b, h, w, c)`: four axes in, four out — permutation or
   reduction?
2. Name the input axes meaningfully (`bchw`) and write the output in the
   required order.
3. Every letter kept on both sides — nothing summed, values untouched:
   `'bchw->bhwc'`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return it in channels-LAST layout (b, h, w, c) — a pure axis permutation, no values changed."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(271)


In [ ]:
#@title 💡 Solution — Problem 271
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('bchw->bhwc', x)


print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q300 -->

### Problem 300 · independent

Write a function solve(img) that takes a channels-first image of shape (c, h, w) and returns the (c, h*w) tensor with the two spatial axes FLATTENED into one, channels preserved — each channel becomes one long row.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])
```


In [ ]:
import torch as t

def solve(img):
    """Return the (c, h*w) array with the two spatial axes FLATTENED into one, channels preserved — each channel beco"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(300)


In [ ]:
#@title 💡 Solution — Problem 300
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(img):
    return img.reshape(img.shape[0], -1)


print(solve(t.arange(8.0).reshape(2, 2, 2)))


<!-- dd:dd-q303 -->

### Problem 303 · independent

Write a function solve(a) that takes a square matrix and returns its ANTISYMMETRIC part a - a.T: entry [i, j] becomes a[i, j] - a[j, i], making the result equal to the negative of its own transpose with a zero diagonal.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 0., -3.],
        [ 3.,  0.]])
```


In [ ]:
import torch as t

def solve(a):
    """Return its ANTISYMMETRIC part a - a.T."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [5.0, 3.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(303)


In [ ]:
#@title 💡 Solution — Problem 303
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return a - a.T


print(solve(t.tensor([[1.0, 2.0], [5.0, 3.0]])))


<!-- dd:dd-kp-einsum-reductions -->

## Sums as index removal

`einsum.reductions`


### One missing index sums one axis


Core rule: **a letter missing from output is summed over**.

For a matrix named `ij`, `'ij->j'` keeps `j` and deletes `i`. Each surviving
`j` therefore receives sum across `i`: one total per column. Conversely,
`'ij->i'` keeps `i` and sums across `j`: one total per row.

Read spec by asking which letters survive. Never memorize visual directions.


> **Watch out.** - **"`'ij->i'` sums the i axis."** — `i` survives. Missing `j` is summed.
- **"einsum only does products."** — One-input specs with missing output
  letters are plain sums. For example, `'ij->'` sums both axes.


Column sums keep `j`:


In [ ]:
import torch as t

a = t.tensor([[1, 2, 3],
              [10, 20, 30]])

column_sums = t.einsum('ij->j', a)
assert column_sums.tolist() == [11, 22, 33]
assert t.equal(column_sums, a.sum(dim=0))
print(a)
print("'ij->j' drops i ->", column_sums, "| a.sum(dim=0) ->", a.sum(dim=0))




`j` stays in output, so result has one value per column. Missing `i` is axis
being collapsed.


<!-- dd:dd-q274 -->

### Problem 274 · faded — your turn

Now reverse survivor: return one total per row.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3, 7])
```


In [ ]:
import torch as t

def solve(a):
    """Return one sum per row."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1, 2], [3, 4]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(274)


In [ ]:
#@title 💡 Solution — Problem 274
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('ij->i', a)


print(solve(t.tensor([[1, 2], [3, 4]])))


### One deletion works same way in higher dimensions


Rank does not change rule. In `'btf->bf'`, `b` and `f` survive while `t`
disappears. Result keeps batch and feature positions, summing only time.


> **Watch out.** - **"A 3-D or 4-D tensor needs axis numbers."** — Same missing-letter rule
  works at any rank. Keep every axis you do not want summed.


Sum time from a `(batch, time, feature)` tensor:


In [ ]:
import torch as t

x = t.arange(12).reshape(2, 3, 2)
per_batch_feature = t.einsum('btf->bf', x)
assert t.equal(per_batch_feature, x.sum(dim=1))
assert per_batch_feature.shape == (2, 2)
print("'btf->bf':", tuple(x.shape), "-> ", tuple(per_batch_feature.shape),
      " (t was the missing letter)")
print(per_batch_feature)




Only `t` disappears, so only time is reduced.


<!-- dd:dd-q280 -->

### Problem 280 · faded — your turn

Given `(b, h, s, d)`, delete only head axis `h`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[2., 2.]]])
```


In [ ]:
import torch as t

def solve(a):
    """Sum h; keep b, s, d."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 1, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(280)


In [ ]:
#@title 💡 Solution — Problem 280
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('bhsd->bsd', a)


print(solve(t.ones((1, 2, 1, 2))))


### Several missing indices sum several axes


Multiple deletions perform multi-axis reduction in one spec. For video tensor
`btchw`, `'btchw->bc'` keeps batch and channel while summing time, height, and
width together.


> **Watch out.** - **"Only one missing letter can be summed."** — Every missing letter is
  summed. Three missing letters mean three collapsed axes.


Keep batch and channel; total time and spatial positions:


In [ ]:
import torch as t

video = t.arange(32).reshape(2, 2, 2, 2, 2)
per_batch_channel = t.einsum('btchw->bc', video)
assert t.equal(per_batch_channel, video.sum(dim=(1, 3, 4)))
assert per_batch_channel.shape == (2, 2)
print("'btchw->bc' drops three axes at once:", tuple(video.shape), "->",
      tuple(per_batch_channel.shape))
print(per_batch_channel)
print("the dim= spelling needs you to count: dim=(1, 3, 4)")




`b` and `c` survive. Missing `t`, `h`, and `w` all collapse.


<!-- dd:dd-q295 -->

### Problem 295 · faded — your turn

Keep batch only; total channels and both spatial axes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 4.])
```


In [ ]:
import torch as t

def solve(x):
    """Return one total per batch item."""
    return t.einsum('_____', x)


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 1, 2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(295)


In [ ]:
#@title 💡 Solution — Problem 295
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('bchw->b', x)


print(solve(t.ones((2, 1, 2, 2))))


### Mean equals einsum sum divided by collapsed length


Einsum sums; it does not divide. To compute mean, first delete desired index,
then divide by that axis's length. For `(b, t, d)`, `'btd->bd'` sums `t`; divide
by `data.shape[1]` because axis 1 is `t`.


> **Watch out.** - **"Einsum has special mean notation."** — No. Division stays outside spec.
- **"Any shape length works as denominator."** — Divide by length of exactly
  collapsed axis or product of lengths when several axes collapse.


Mean each matrix column by summing rows, then dividing by row count:


In [ ]:
import torch as t

a = t.tensor([[1.0, 3.0],
              [5.0, 7.0]])
column_means = t.einsum('ij->j', a) / a.shape[0]
assert t.allclose(column_means, a.mean(dim=0))
print("einsum sums; the division is yours to write:")
print("sum", t.einsum('ij->j', a), "/", a.shape[0], "->", column_means)




Missing `i` performs column sums; `a.shape[0]` converts those sums to means.


<!-- dd:dd-q282 -->

### Problem 282 · faded — your turn

Mean over middle axis `t`: choose sum spec and denominator.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 1.]])
```


In [ ]:
import torch as t

def solve(data):
    """Return mean over t from shape (b, t, d)."""
    return t.einsum('_____', data) / data.shape[_____]


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 4, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(282)


In [ ]:
#@title 💡 Solution — Problem 282
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(data):
    return t.einsum('btd->bd', data) / data.shape[1]


print(solve(t.ones((1, 4, 2))))


<!-- dd:dd-q247 -->

### Problem 247 · guided

Write a function solve(a) that takes a 2-D PyTorch tensor and returns a 1-D tensor of its COLUMN SUMS: entry j is the total of column j (the row axis is collapsed).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4, 6])
```


<details>
<summary>Hints</summary>

1. Column sums keep column index `j`.
2. Delete row index `i` from `'ij->?'`.
3. Check against `a.sum(axis=0)` mentally.

</details>


In [ ]:
import torch as t

def solve(a):
    """Return the sum of each column of a."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 2], [3, 4]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(247)


In [ ]:
#@title 💡 Solution — Problem 247
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('ij->j', a)


print(solve(t.tensor([[1, 2], [3, 4]])))


<!-- dd:dd-q289 -->

### Problem 289 · independent

Write a function solve(x) that takes a 3-D tensor of shape (b, t, d) and returns the (t, d) tensor summing over the BATCH axis: each (t, d) cell aggregates that position's values across all batch items.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[3., 3.]])
```


In [ ]:
import torch as t

def solve(x):
    """Return the (t, d) array summing over the BATCH axis."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((3, 1, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(289)


In [ ]:
#@title 💡 Solution — Problem 289
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('btd->td', x)


print(solve(t.ones((3, 1, 2))))


<!-- dd:dd-q249 -->

### Problem 249 · independent

Write a function solve(x) that takes a 4-D float tensor of shape (b, c, h, w) — a batch of multi-channel images — and returns a 1-D tensor of length b where entry n is the SUM OF SQUARES of every value in image n (across all channels and pixels).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 4.])
```


In [ ]:
import torch as t

def solve(x):
    """Return per-image sums of squared values for the batch x."""
    return None


# Example run — the grader calls solve() with several batches.
print(solve(t.ones((2, 1, 2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(249)


In [ ]:
#@title 💡 Solution — Problem 249
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('bchw,bchw->b', x, x)


print(solve(t.ones((2, 1, 2, 2))))


<!-- dd:dd-q261 -->

### Problem 261 · independent

Write a function solve(x) that takes a PyTorch tensor of ANY shape and returns the total of all its elements as a scalar, expressed as a full contraction (every index summed away).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(15.)
```


In [ ]:
import torch as t

def solve(x):
    """Return the total of all its elements as a scalar, expressed as a full contraction (every index summed away)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(261)


In [ ]:
#@title 💡 Solution — Problem 261
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum(x, list(range(x.ndim)), [])


print(solve(t.arange(6.0).reshape(2, 3)))


<!-- dd:dd-q290 -->

### Problem 290 · independent

Write a function solve(x) that takes a PyTorch tensor of ANY rank and returns its MEAN over all elements as a scalar — using a rank-generic einsum contraction (sum every axis), then dividing by the element count.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(2.5000)
```


In [ ]:
import torch as t

def solve(x):
    """Return its MEAN over all elements as a scalar — using einsum's ellipsis '...' to sum without knowing the rank,"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(290)


In [ ]:
#@title 💡 Solution — Problem 290
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    total = t.einsum(x, list(range(x.ndim)), [])
    return total / x.numel()


print(solve(t.arange(6.0).reshape(2, 3)))


<!-- dd:dd-q302 -->

### Problem 302 · independent

Write a function solve(v) that takes a 5-D video tensor of shape (b, t, c, h, w) and returns the (b, c) tensor holding the MEAN over time and both spatial axes — sum out t, h, w and divide by their combined count.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 1., 1.]])
```


In [ ]:
import torch as t

def solve(v):
    """Return the (b, c) array holding the MEAN over time and both spatial axes — sum out t, h, w and divide by their"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 3, 2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(302)


In [ ]:
#@title 💡 Solution — Problem 302
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    return t.einsum('btchw->bc', v) / (v.shape[1] * v.shape[3] * v.shape[4])


print(solve(t.ones((1, 2, 3, 2, 2))))


<!-- dd:dd-kp-einsum-dot-frobenius -->

## Dot and Frobenius inner products

`einsum.dot-frobenius`


### the dot product 'i,i->'


So far each spec had ONE input. With two inputs, one new rule switches on:

> **A letter shared between two inputs pairs those axes elementwise** — the
> operands get multiplied along it.

Combine that with the rule you already know — a letter missing from the output
is summed away — and a letter that is *shared then dropped* means "multiply
corresponding entries, then add them up." That is exactly the dot product:

- `'i,i->'` — both vectors carry `i`, and `i` is absent from the output.
  So: pair entry `i` with entry `i`, multiply, sum. A single scalar falls out.

The empty right side (`->` with nothing after it) is what makes the answer a
scalar — every letter was summed, none survived.

This is the whole foundation: everything later (matvec, matmul, batching,
attention) is only ever combinations of these two rules — **shared letter =
multiply along it; dropped letter = sum over it.**


> **Watch out.** - **"`'i,i->'` is illegal because `i` is used twice."** — No. Sharing a letter
  ACROSS two inputs is the multiplication mechanism, not a name clash. (A letter
  repeated WITHIN one input, like `'ii'`, is a different legal thing — the
  diagonal — which comes later.)


In [ ]:
import torch as t

v1 = t.tensor([1.0, 2.0, 3.0])
v2 = t.tensor([4.0, -5.0, 6.0])

# 'i,i->': shared i pairs the entries; the empty output sums the products.
d = t.einsum('i,i->', v1, v2)
assert d == 12.0                              # 1*4 + 2*-5 + 3*6
assert d == t.dot(v1, v2) == (v1 * v2).sum() # three spellings, one atom
print("paired products:", v1 * v2)
print("'i,i->' ->", d.item(), "| t.dot ->", t.dot(v1, v2).item())




Why: `1·4 + 2·(-5) + 3·6 = 4 - 10 + 18 = 12`. The three-spellings assert is the
bridge — when a spec confuses you, translate it back to "multiply the paired
axis, then sum" and it parses. `t.dot`, `(v1*v2).sum()`, and `'i,i->'` are the
same operation wearing three different clothes.


<!-- dd:dd-q245 -->

### Problem 245 · faded — your turn

The dot product of two vectors. There's no visible matrix or `.sum()` — just two
vectors and a scalar answer. Name the shared axis, then leave the output empty so
it collapses to a scalar.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(11.)
```


In [ ]:
import torch as t

def solve(v1, v2):
    """Dot product as a scalar: pair the shared axis, sum it away."""
    return t.einsum('_____', v1, v2)


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(245)


In [ ]:
#@title 💡 Solution — Problem 245
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v1, v2):
    return t.einsum('i,i->', v1, v2)


print(solve(t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0])))


### the Frobenius inner product 'ij,ij->'


The dot generalizes with no new rule — just more shared letters. Two matrices of
the SAME shape have two axes to pair, `i` and `j`:

- `'ij,ij->'` — position `[i, j]` on the first input meets its OWN position
  `[i, j]` on the second. Multiply every corresponding pair, sum them all. The
  result is a single scalar: the **Frobenius inner product**.

It's the dot product with one extra letter — the same "pair the shared axes,
drop them to sum" move, now over a 2-D grid instead of a line. (And its cousin:
dot a matrix with *itself* and square-root the result — `sqrt('ij,ij->')` — and
you have the Frobenius norm. Same spec, nothing new to learn.)


> **Watch out.** - **`'ij,ij'` is NOT `'ij,ji'`.** `'ij,ij->'` pairs each position with its own
  coordinates (no transpose). `'ij,ji->'` pairs `[i,j]` with `[j,i]` — a
  transposed pairing that computes `tr(a @ b)`, a completely different number.
  Read the letters exactly; don't pattern-match on "two matrices → one scalar."


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# 'ij,ij->': position [i,j] meets [i,j] on both; multiply all, sum all.
f = t.einsum('ij,ij->', a, b)
assert f == 70.0                 # 1*5 + 2*6 + 3*7 + 4*8
assert f == (a * b).sum()        # the same multiply-then-reduce, named
print("a * b, before the sum:")
print(a * b)
print("'ij,ij->' ->", f.item())




Why: hand-compute one term — `1·5 = 5` at position [0,0], `2·6 = 12` at [0,1],
and so on: `5 + 12 + 21 + 32 = 70`. `ij` on BOTH inputs means [i,j] meets [i,j] —
no cross terms, no transpose. It is the dot rule with a second shared letter.


<!-- dd:dd-q267 -->

### Problem 267 · faded — your turn

The second matrix `b` is all ones. Multiplying by 1 changes nothing, so this
*looks* like "just add up every entry of `a`" — and it is. But get there through
the Frobenius spec: it's still the elementwise-multiply-and-sum of two matrices,
`a` paired position-for-position with a matrix of ones.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(10.)
```


In [ ]:
import torch as t

def solve(a, b):
    """Sum of a*b over every corresponding entry (b is all ones here)."""
    return t.einsum('_____', a, b)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.ones((2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(267)


In [ ]:
#@title 💡 Solution — Problem 267
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,ij->', a, b)


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.ones((2, 2))))


<!-- dd:dd-q270 -->

### Problem 270 · guided

Write a function solve(a, b) that takes two 3-D tensors of identical shape and returns the SCALAR total of the products of corresponding elements — the full elementwise contraction across all three axes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(8.)
```


<details>
<summary>Hints</summary>

1. Sum of products of corresponding elements of two SAME-SHAPE rank-3 tensors —
   how many axes need pairing now?
2. Three axes → three shared letters; the scalar answer means the output side is
   empty.
3. `'abc,abc->'` — the dot pattern generalizes by adding one letter per axis,
   nothing else changes.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the SCALAR total of the products of corresponding elements — the full elementwise contraction across al"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 2, 2)), t.ones((2, 2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(270)


In [ ]:
#@title 💡 Solution — Problem 270
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('abc,abc->', a, b)


print(solve(t.ones((2, 2, 2)), t.ones((2, 2, 2))))


<!-- dd:dd-q308 -->

### Problem 308 · independent

Write a function solve(x) that takes a sequence x of shape (t, d) with t >= 2 and returns the length-(t-1) tensor whose entry i is the DOT PRODUCT of token i with the NEXT token i+1 — adjacent-pair similarities along the sequence.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2., 3.])
```


In [ ]:
import torch as t

def solve(x):
    """Return the length-(t-1) array whose entry i is the DOT PRODUCT of token i with the NEXT token i+1 — adjacent-p"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 0.0], [2.0, 3.0], [0.0, 1.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(308)


In [ ]:
#@title 💡 Solution — Problem 308
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('td,td->t', x[:-1], x[1:])


print(solve(t.tensor([[1.0, 0.0], [2.0, 3.0], [0.0, 1.0]])))


<!-- dd:dd-kp-einsum-outer-products -->

## Outer products — new axes from free indices

`einsum.outer-products`


The dot product shared its letter. The **outer product** is the opposite
move: give each input its OWN letter and keep both —

- `'i,j->ij'` — no shared letters, nothing dropped. Every element of v1
  meets every element of v2 exactly once: entry [p, q] = v1[p] · v2[q].
  The (i, j) output is the multiplication table of the two vectors.

The general reading: **unshared kept letters multiply COMBINATORIALLY** —
the output ranges over all combinations, which is why the result's rank is
the sum of the input ranks. That single idea covers the family:

- **Self outer product**: pass the same vector twice — `'i,j->ij'` with
  (v, v) gives the symmetric all-pairs product matrix. (The two slots get
  different letters even though the DATA is the same tensor — letters label
  axes-of-slots, not variables.)
- **Tensor outer product of matrices**: `'pq,rs->prqs'` — four independent
  axes, kept in whatever output order the task demands. Axis order on the
  right is yours to choose; `(p, r, q, s)` interleaves the two inputs' axes.
- **Mixed forms**: share SOME letters, keep others —
  `'id,jd->ijd'` pairs the d axis elementwise (shared, kept!) while i and j
  combine combinatorially: entry [i, j] is the elementwise product of row i
  and row j. Shared-and-kept means "multiply along it but DON'T sum" — the
  third notational possibility, completing the set: shared+dropped = dot,
  unshared+kept = outer, shared+kept = elementwise.

Broadcasting connection: `'i,j->ij'` computes exactly
`v1[:, None] * v2[None, :]` — einsum is naming what the None-insertion
pattern built by hand in np-3.


Task: an outer product, its self- variant, and the shared-and-kept mixed
form.


In [ ]:
import torch as t

v1 = t.tensor([1.0, 2.0])
v2 = t.tensor([10.0, 20.0, 30.0])

# 'i,j->ij': independent letters, both kept -> all pairs.
outer = t.einsum('i,j->ij', v1, v2)
assert outer.shape == (2, 3)
assert outer.tolist() == [[10.0, 20.0, 30.0],
                          [20.0, 40.0, 60.0]]
# Same thing via broadcasting — einsum names this exact pattern:
assert t.equal(outer, v1[:, None] * v2[None, :])

# Self outer product: same tensor in both slots, different letters.
v = t.tensor([1.0, 2.0, 3.0])
self_outer = t.einsum('i,j->ij', v, v)
assert self_outer[1, 2] == 6.0                 # v[1] * v[2]
assert t.equal(self_outer, self_outer.T)  # symmetric by construction

# Shared AND kept: 'id,jd->ijd' — d pairs elementwise, i/j combine.
x = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])                     # (t=2, d=2)
pairs = t.einsum('id,jd->ijd', x, x)
assert pairs.shape == (2, 2, 2)
assert pairs[0, 1].tolist() == [3.0, 8.0]      # row0 * row1, elementwise
print("'i,j->ij' — nothing contracted, so both axes survive:")
print(outer)
print("self outer product is symmetric:")
print(self_outer)
print("'id,jd->ijd' — d is shared AND kept:", tuple(pairs.shape))
print(pairs[0, 1])




Why each step:

1. The broadcasting equivalence is the deepest line here: outer products,
   None-insertion, and `'i,j->ij'` are one concept in three notations. If
   you can write any one, you can now write all three.
2. In the self-outer, note WHY two letters: each SLOT of the einsum gets its
   own axis names. Passing v twice with 'i,i->…' would instead pair the
   axes elementwise — a different (and here wrong) computation.
3. The mixed 'id,jd->ijd' is worth slow reading: d shared (multiply along
   it) and kept (don't sum) — so the elementwise products survive as the
   last axis. Dropping d instead ('id,jd->ij') would sum them — turning
   this into the pairwise-dots table. One letter's fate, two different
   drills.


<!-- dd:dd-q256 -->

### Problem 256 · faded — your turn

The (i, j) outer-product matrix of two vectors.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[10., 20., 30.],
        [20., 40., 60.]])
```


In [ ]:
import torch as t

def solve(v1, v2):
    """All pairwise products: independent letters, both kept."""
    return t.einsum('_____', v1, v2)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1.0, 2.0]), t.tensor([10.0, 20.0, 30.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(256)


In [ ]:
#@title 💡 Solution — Problem 256
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v1, v2):
    return t.einsum('i,j->ij', v1, v2)


print(solve(t.tensor([1.0, 2.0]), t.tensor([10.0, 20.0, 30.0])))


<!-- dd:dd-q278 -->

### Problem 278 · guided

Write a function solve(v) that takes a single 1-D vector of length n and returns the (n, n) symmetric matrix of ALL pairwise products of its entries: entry [i, j] is v[i] * v[j] — the vector's outer product with itself.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 2., 3.],
        [2., 4., 6.],
        [3., 6., 9.]])
```


<details>
<summary>Hints</summary>

1. All pairwise products of ONE vector's entries — an outer product of the
   vector with which second operand?
2. The same tensor can fill both slots; the spec doesn't change.
3. `t.einsum('i,j->ij', v, v)` — and the result should equal its own
   transpose (why?).

</details>


In [ ]:
import torch as t

def solve(v):
    """Return the (n, n) symmetric matrix of ALL pairwise products of its entries."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1.0, 2.0, 3.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(278)


In [ ]:
#@title 💡 Solution — Problem 278
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    return t.einsum('i,j->ij', v, v)


print(solve(t.tensor([1.0, 2.0, 3.0])))


<!-- dd:dd-q275 -->

### Problem 275 · independent

Write a function solve(a, b) that takes two 2-D tensors of shapes (p, q) and (r, s) and returns their (p, r, q, s) TENSOR (outer) product with the axes interleaved as p, r, q, s: entry [i, k, j, l] equals a[i, j] * b[k, l].

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[3.],
          [6.]],

         [[4.],
          [8.]]]])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return their (p, r, q, s) TENSOR (outer) product with the axes interleaved as p, r, q, s."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0], [4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(275)


In [ ]:
#@title 💡 Solution — Problem 275
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('pq,rs->prqs', a, b)


print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0], [4.0]])))


<!-- dd:dd-q292 -->

### Problem 292 · independent

Write a function solve(u, v, a) that takes vectors u (length m) and v (length n) and a scalar a, and returns the (m, n) matrix whose entry [i, j] is u[i] * v[j] + a — the outer product shifted by the scalar.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[15.],
        [25.]])
```


In [ ]:
import torch as t

def solve(u, v, a):
    """Return the (m, n) matrix whose entry [i, j] is u[i] * v[j] + a — the outer product shifted by the scalar."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1.0, 2.0]), t.tensor([10.0]), 5.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(292)


In [ ]:
#@title 💡 Solution — Problem 292
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(u, v, a):
    return t.einsum('i,j->ij', u, v) + a


print(solve(t.tensor([1.0, 2.0]), t.tensor([10.0]), 5.0))


<!-- dd:dd-q296 -->

### Problem 296 · independent

Write a function solve(x) that takes a 2-D tensor of shape (t, d) and returns the (t, t, d) tensor whose entry [i, j] holds the ELEMENTWISE product of rows i and j — the per-feature interaction of every pair of rows (d is NOT summed).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 1.,  4.],
         [ 3.,  8.]],

        [[ 3.,  8.],
         [ 9., 16.]]])
```


In [ ]:
import torch as t

def solve(x):
    """Return the (t, t, d) array whose entry [i, j] holds the ELEMENTWISE product of rows i and j — the per-feature """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(296)


In [ ]:
#@title 💡 Solution — Problem 296
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.einsum('id,jd->ijd', x, x)


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]])))


#### Common mistakes

- **"Passing the same tensor twice needs the same letter."** — Letters name
  slot axes, not tensors. Self-outer is 'i,j->ij' with (v, v); using 'i,i'
  would elementwise-pair instead. Decide by the COMPUTATION, not the
  operand identity.
- **"Output rank = input rank."** — Unshared kept letters ADD ranks:
  vectors (1+1) → matrix, matrices (2+2) → 4-D tensor. If the output shape
  surprises you, count the distinct kept letters.
- **"Shared letters are always summed."** — Only if also dropped. Shared +
  KEPT = elementwise multiply along that axis, products retained
  ('id,jd->ijd'). The full decision table is: shared+dropped = contract,
  shared+kept = elementwise, unshared+kept = combinatorial.


<!-- dd:dd-kp-einsum-matvec-matmul -->

## Matrix-vector and matrix-matrix products

`einsum.matvec-matmul`


### matrix-vector 'ij,j->i'


You already know the dot: `'i,i->'` lines two vectors up on a shared letter,
multiplies, and sums it away. A matrix-vector product is that SAME move, done
once per row.

Label the axes. A matrix is `(i, j)`; a vector is `(j,)`. Write them side by
side: `'ij,j->i'`. Now read each letter:

- **j appears in BOTH inputs** — it's the shared letter. einsum sums over any
  letter that is shared and does not appear in the output. So j is multiplied
  aligned and summed away. That summing IS a dot: for a fixed row, `Σ_j a[i,j]·v[j]`
  is row i dotted with v.
- **i appears only in the matrix, and in the output** — it's private and kept.
  It picks WHICH row. One surviving letter → one number per row.

So `'ij,j->i'` says: for each row i, dot that row with v. The notation didn't
memorize "matrix times vector" — it fell out of "sum the shared letter, keep
the private one."


> **Watch out.** - **Which axis is shared depends on the shapes, not habit.** A *vector times a
  matrix* (`v @ M`, shapes `(i,)` and `(i,j)`) shares the FIRST matrix axis:
  `'i,ij->j'`. Don't reflexively write `'ij,j->i'` — check which axis the
  vector's length lines up with, then make THAT the shared letter.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
v = t.tensor([10.0, 1.0])

# 'ij,j->i': j is shared+dropped (row . v); i is private+kept (one per row).
mv = t.einsum('ij,j->i', a, v)
assert mv.tolist() == [12.0, 34.0]
assert t.equal(mv, a @ v)   # the @-twin: always check while learning
print("'ij,j->i' ->", mv, "| a @ v ->", a @ v)




Why: trace entry [0] straight from the rule — i=0 fixed, j ranges:
`Σ_j a[0,j]·v[j] = 1·10 + 2·1 = 12`. That is row 0 dotted with v. Entry [1] is
row 1 dotted with v: `3·10 + 4·1 = 34`. The result is one dot per row — nothing
memorized beyond "sum the shared letter."


<!-- dd:dd-q312 -->

### Problem 312 · faded — your turn

A permutation matrix `p` (each row is a single 1, the rest 0) times a vector
`v`. It looks like it *shuffles* v rather than dotting anything — but a
permutation matrix is still a matrix, so this is still a matrix-vector product.
Write the spec.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([20., 10.])
```


In [ ]:
import torch as t

def solve(p, v):
    """p @ v — each row of p has one 1, so row i just picks out one entry of v."""
    return t.einsum('_____', p, v)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[0.0, 1.0], [1.0, 0.0]]), t.tensor([10.0, 20.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(312)


In [ ]:
#@title 💡 Solution — Problem 312
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(p, v):
    return t.einsum('ij,j->i', p, v)


print(solve(t.tensor([[0.0, 1.0], [1.0, 0.0]]), t.tensor([10.0, 20.0])))


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
v = t.tensor([10.0, 1.0])

# 'ij,j->i': j is shared+dropped (row . v); i is private+kept (one per row).
mv = t.einsum('ij,j->i', a, v)
assert mv.tolist() == [12.0, 34.0]
assert t.equal(mv, a @ v)   # the @-twin: always check while learning
print("'ij,j->i' ->", mv, "| a @ v ->", a @ v)




Why: trace entry [0] straight from the rule — i=0 fixed, j ranges:
`Σ_j a[0,j]·v[j] = 1·10 + 2·1 = 12`. That is row 0 dotted with v. Entry [1] is
row 1 dotted with v: `3·10 + 4·1 = 34`. The result is one dot per row — nothing
memorized beyond "sum the shared letter."


<!-- dd:dd-q286 -->

### Problem 286 · faded — your turn

Same contraction, arguments the other way round: a single query vector `q` of
length d against a memory matrix `m` of shape (n, d), giving one similarity
per memory row. The example's spec will not transfer letter for letter — the
vector is the FIRST operand here, and the shared axis is d, not the matrix's
first axis. Name the axes of each operand as they actually are, then keep the
one the output needs.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2., 0.])
```


In [ ]:
import torch as t

def solve(q, m):
    """Length-n vector: entry i is the dot product of q with row i of m."""
    return t.einsum('_____', q, m)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1.0, 0.0]), t.tensor([[2.0, 5.0], [0.0, 3.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(286)


In [ ]:
#@title 💡 Solution — Problem 286
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(q, m):
    return t.einsum('d,nd->n', q, m)


print(solve(t.tensor([1.0, 0.0]), t.tensor([[2.0, 5.0], [0.0, 3.0]])))


### matrix-matrix 'ik,kj->ij'


Matmul is a matrix-vector product done once per *column* of the second matrix —
i.e. a whole grid of dots, one for every (row, column) pair. The notation says
exactly that.

Label the axes: A is `(i, k)`, B is `(k, j)`. Write them: `'ik,kj->ij'`. Read
the letters:

- **k appears in both inputs and NOT in the output** — shared and dropped, so
  einsum sums over it. k is the "inner" axis, the one A and B have in common; it
  gets contracted away. `Σ_k a[i,k]·b[k,j]` is row i of A dotted with column j
  of B.
- **i appears only in A and survives** — it picks the row.
- **j appears only in B and survives** — it picks the column.

Two survivors → a 2-D result, indexed `[i, j]`. So
`result[i, j] = Σ_k a[i,k]·b[k,j]` = "row i of A meets column j of B", for every
i and j. That's the definition of matmul, read straight off the spec.

The notation even encodes the shape rule: the shared letter k is the inner
dimension that must match and then vanishes; the two survivors i and j become
the output shape `(i, j)`. `(m,k) @ (k,n) → (m,n)` is just "drop the shared
letter, keep the private ones."


> **Watch out.** - **One transposed letter is a silent, wrong matmul.** `'ik,kj->ij'` contracts
  B's FIRST axis (correct). `'ik,jk->ij'` contracts B's SECOND axis — that's
  `a @ b.T`. Same output shape, different numbers, no error raised. This is THE
  einsum bug. The `@`-twin assert is how you catch it while learning.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# 'ik,kj->ij': k is shared+dropped (the contraction); i, j are private+kept.
mm = t.einsum('ik,kj->ij', a, b)
assert t.equal(mm, a @ b)   # the @-twin
print("'ik,kj->ij' — k contracted away, i and j kept:")
print(mm)




Why: trace entry [0, 0] by the rule — i=0, j=0 fixed, k ranges:
`Σ_k a[0,k]·b[k,0] = 1·5 + 2·7 = 19` = row 0 of A dotted with column 0 of B.
Entry [1, 0] uses row 1 and column 0: `3·5 + 4·7 = 43`. Each output cell is one
row-dot-column; matmul is the whole grid of them.


<!-- dd:dd-q260 -->

### Problem 260 · faded — your turn

Matrix product of `c` with shape `(1, 2)` and `d` with shape `(2, 1)`. The
result is `(1, 1)` — a single number in a 2-D box. It *looks* like a plain dot
of a row and a column, so it's tempting to write `'i,i->'`. But both inputs are
2-D matrices, so it's a matmul: name every axis and let the survivors decide the
output shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[11.]])
```


In [ ]:
import torch as t

def solve(c, d):
    """(1,2) @ (2,1) -> (1,1). Two matrices -> matmul, not a bare dot."""
    return t.einsum('_____', c, d)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0], [4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(260)


In [ ]:
#@title 💡 Solution — Problem 260
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c, d):
    return t.einsum('ik,kj->ij', c, d)


print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0], [4.0]])))


<!-- dd:dd-q262 -->

### Problem 262 · guided

Write a function solve(a, b) that takes an (i, j) matrix and a length-j vector, and returns the length-i MATRIX-VECTOR product: entry i is the dot product of row i of a with b.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([12., 34.])
```


<details>
<summary>Hints</summary>

1. Two inputs: a matrix with axes `(i, j)` and a vector with axis `(j,)`. Which
   letter do they share?
2. The shared letter j is summed away (each row dots the vector); i is private
   to the matrix and survives, one result per row.
3. `'ij,j->i'`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the length-i MATRIX-VECTOR product."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 1.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(262)


In [ ]:
#@title 💡 Solution — Problem 262
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,j->i', a, b)


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 1.0])))


<!-- dd:dd-q264 -->

### Problem 264 · independent

Write a function solve(a, b, c) that takes three matrices with chain-compatible shapes (i, k), (k, m), (m, p) and returns the triple MATRIX PRODUCT a @ b @ c as one einsum contraction.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 2.],
        [3., 4.]])
```


In [ ]:
import torch as t

def solve(a, b, c):
    """Return the triple MATRIX PRODUCT a @ b @ c as one einsum contraction."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.eye(2), t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.eye(2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(264)


In [ ]:
#@title 💡 Solution — Problem 264
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b, c):
    return t.einsum('ik,km,mp->ip', a, b, c)


print(solve(t.eye(2), t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.eye(2)))


<!-- dd:dd-q311 -->

### Problem 311 · independent

Write a function solve(a, v, b) that takes an (i, j) matrix a, a length-j vector v representing a DIAGONAL matrix, and a (j, k) matrix b, and returns the (i, k) product a @ diag(v) @ b — without ever materializing the diagonal matrix.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2., 0.],
        [0., 3.]])
```


In [ ]:
import torch as t

def solve(a, v, b):
    """Return the (i, k) product a @ diag(v) @ b — without ever materializing the diagonal matrix."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.eye(2), t.tensor([2.0, 3.0]), t.eye(2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(311)


In [ ]:
#@title 💡 Solution — Problem 311
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, v, b):
    return t.einsum('ij,j,jk->ik', a, v, b)


print(solve(t.eye(2), t.tensor([2.0, 3.0]), t.eye(2)))


<!-- dd:dd-q294 -->

### Problem 294 · independent

Write a function solve(w, x) that takes a weight tensor w of shape (o, i, h, wd) and an input tensor x of shape (i, h, wd), and returns the length-o vector contracting x against each output filter — every axis of x pairs with the matching axis of w.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 4.])
```


In [ ]:
import torch as t

def solve(w, x):
    """Return the length-o vector contracting x against each output filter — every axis of x pairs with the matching """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 1, 2, 2)), t.ones((1, 2, 2))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(294)


In [ ]:
#@title 💡 Solution — Problem 294
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w, x):
    return t.einsum('oihw,ihw->o', w, x)


print(solve(t.ones((2, 1, 2, 2)), t.ones((1, 2, 2))))


<!-- dd:dd-kp-einsum-diag-trace -->

## Repeated indices on one operand — diagonal and trace

`einsum.diag-trace`


### 'ii->i' — a repeated index walks the diagonal


A letter repeated WITHIN a single operand makes those two axes move
**together** — einsum visits only the entries where their indices are equal.

`'ii->i'`: both of the matrix's axes are named `i`, so only entries `a[i, i]`
are visited; keeping `i` in the output emits them. That's the **diagonal**,
as a vector.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])

# 'ii->i': both axes locked together -> visit a[0,0], a[1,1]; keep them.
diag = t.einsum('ii->i', a)
assert diag.tolist() == [1.0, 4.0]
assert t.equal(diag, t.diag(a))
print(a)
print("'ii->i' ->", diag, "| t.diag ->", t.diag(a))




Why: the repeated `i` is the whole trick — it selects the diagonal. Keeping
`i` on the output side means "emit what you selected".


<!-- dd:dd-q257 -->

### Problem 257 · faded — your turn

The main diagonal, as a vector.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 4])
```


In [ ]:
import torch as t

def solve(a):
    """Diagonal: walk it, keep it."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1, 2], [3, 4]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(257)


In [ ]:
#@title 💡 Solution — Problem 257
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('ii->i', a)


print(solve(t.tensor([[1, 2], [3, 4]])))


### 'ii->' — drop the index to sum it (the trace)


Same selection as before — walk the diagonal — but now **drop** `i` from the
output. A letter dropped from the output is summed. So `'ii->'` selects the
diagonal, then sums it: the **trace**, a scalar.

Read every spec as selection + fate: `'ii->i'` and `'ii->'` select the same
entries; the only difference is keep (`->i`) vs sum (`->`).


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])

# 'ii->': same diagonal walk, then sum -> the trace.
tr = t.einsum('ii->', a)
assert tr == 5.0
assert tr == t.trace(a)
print("'ii->i' keeps the walk:", t.einsum('ii->i', a))
print("'ii->'  sums it:      ", tr, "| t.trace ->", t.trace(a))




Why: one character (`->i` vs `->`) flips "emit the diagonal" into "sum the
diagonal". That's the payoff of reading specs as selection + fate.


<!-- dd:dd-q277 -->

### Problem 277 · faded — your turn

The trace, via the repeated-index convention.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(5.)
```


In [ ]:
import torch as t

def solve(a):
    """Trace: walk the diagonal, sum it."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(277)


In [ ]:
#@title 💡 Solution — Problem 277
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('ii->', a)


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]])))


### 'bii->bi' — a batch letter rides along


Add one more letter for a batch axis and the diagonal trick works per
matrix. For a stack of shape `(b, n, n)`, `'bii->bi'` extracts each matrix's
diagonal: the repeated `i` still walks each matrix's diagonal, and `b` is an
ordinary kept axis carried straight through.

No new function exists for "diagonal of every matrix in a stack" — the spec
is one letter away from the single-matrix version.


In [ ]:
import torch as t

stack = t.stack([t.diag(t.tensor([1.0, 2.0])),
                  t.diag(t.tensor([3.0, 4.0]))])   # shape (2, 2, 2)

# 'bii->bi': per-matrix diagonal; b rides along untouched.
diags = t.einsum('bii->bi', stack)
assert diags.tolist() == [[1.0, 2.0], [3.0, 4.0]]
print("stack", tuple(stack.shape), "-> 'bii->bi'", tuple(diags.shape))
print(diags)




Why: `b` is kept and unpaired, so it just indexes the batch; the `ii` does
the same diagonal work inside each slice.


<!-- dd:dd-q273 -->

### Problem 273 · faded — your turn

The diagonal of each matrix in a batch — shape (b, n).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 2.],
        [3., 4.]])
```


In [ ]:
import torch as t

def solve(a):
    """Batched diagonal: (b, n, n) -> (b, n)."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several inputs.
print(solve(t.stack([t.diag(t.tensor([1.0, 2.0])), t.diag(t.tensor([3.0, 4.0]))])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(273)


In [ ]:
#@title 💡 Solution — Problem 273
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('bii->bi', a)


print(solve(t.stack([t.diag(t.tensor([1.0, 2.0])), t.diag(t.tensor([3.0, 4.0]))])))


### 'bii->b' — batch trace (drop i, keep b)


Combine the two ideas: keep the batch letter, drop the diagonal letter.
`'bii->b'` sums each matrix's diagonal and keeps one scalar per batch entry —
the **trace of every matrix in the stack**.


In [ ]:
import torch as t

stack = t.stack([t.eye(2), 3 * t.eye(2)])   # traces 2 and 6

# 'bii->b': per-matrix trace; b kept, i dropped (summed).
traces = t.einsum('bii->b', stack)
assert traces.tolist() == [2.0, 6.0]
print("'bii->bi' keeps i:", t.einsum('bii->bi', stack).tolist())
print("'bii->b'  sums it:", traces)




Why: `b` kept + `i` dropped = "one summed diagonal per batch slice". Same
selection as `bii->bi`, different fate for `i`.


<!-- dd:dd-q266 -->

### Problem 266 · faded — your turn

The trace of each matrix in a batch — length b.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2., 6.])
```


In [ ]:
import torch as t

def solve(a):
    """Batched trace: (b, n, n) -> (b,)."""
    return t.einsum('_____', a)


# Example run — the grader calls solve() with several inputs.
print(solve(t.stack([t.eye(2), 3 * t.eye(2)])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(266)


In [ ]:
#@title 💡 Solution — Problem 266
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.einsum('bii->b', a)


print(solve(t.stack([t.eye(2), 3 * t.eye(2)])))


### 'ik,ki->i' — diagonal of a product, without the product


The diagonal of `a @ b` is `Σₖ a[i,k]·b[k,i]`. Write exactly that:
`'ik,ki->i'`. The shared `k` contracts (summed); `i` appears in BOTH operands'
outer positions and survives, forcing row `i` of `a` to pair with column `i`
of `b`. Only the `n` diagonal dots are computed — O(n²), not the O(n³) full
product.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# 'ik,ki->i': entry i = row i of a . column i of b.
dprod = t.einsum('ik,ki->i', a, b)
assert t.equal(dprod, t.diag(a @ b))   # verified vs the slow way
print("a @ b in full — we only wanted its diagonal:")
print(a @ b)
print("'ik,ki->i' computes just that:", dprod)




Why: entry 0 is `Σₖ a[0,k]·b[k,0]` — row 0 of `a` dotted with column 0 of
`b`, i.e. `(a@b)[0,0]`. The spec computes only the n needed dots.


<!-- dd:dd-q246 -->

### Problem 246 · faded — your turn

Diagonal of a @ b, computed directly (no full product).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([19., 50.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Diagonal of a @ b: entry i = row i of a . column i of b."""
    return t.einsum('_____', a, b)


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(246)


In [ ]:
#@title 💡 Solution — Problem 246
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ik,ki->i', a, b)


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]])))


### 'ij,ji->' — trace of a product


Drop the surviving `i` too and you sum the whole diagonal of the product:
`'ij,ji->'` computes `tr(a @ b)` directly. Note the letter positions:
`'ij,ji->'` transposes the second operand's axes relative to the Frobenius
spec `'ij,ij->'`. So `tr(a@b) = Frobenius(a, b.T)` — the notation makes the
identity visible. Read letter POSITIONS, not letter SETS.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# 'ij,ji->': sum over the whole diagonal of a @ b.
assert t.einsum('ij,ji->', a, b) == t.trace(a @ b)
print("'ij,ji->' ->", t.einsum('ij,ji->', a, b).item(),
      "| t.trace(a @ b) ->", t.trace(a @ b).item())




Why: `'ij,ji->'` pairs `a[i,j]` with `b[j,i]` and sums everything — exactly
`Σᵢ (a@b)[i,i]`. The transposed second operand is what separates it from the
Frobenius product.


<!-- dd:dd-q298 -->

### Problem 298 · faded — your turn

Trace of a @ b as one contraction.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(5.)
```


In [ ]:
import torch as t

def solve(a, b):
    """tr(a @ b) directly — one einsum, no full product."""
    return t.einsum('_____', a, b)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.eye(2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(298)


In [ ]:
#@title 💡 Solution — Problem 298
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,ji->', a, b)


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.eye(2)))


<!-- dd:dd-q293 -->

### Problem 293 · guided

Write a function solve(a) that takes a batch of matrices of shape (b, n, n) and returns the 1-D length-b tensor of each matrix's TOP-LEFT element a[k, 0, 0].

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0., 4.])
```


<details>
<summary>Hints</summary>

1. Read the target carefully: a[k, 0, 0] fixes two indices at a CONSTANT.
   Einsum's subscripts can repeat an index or drop it, but they cannot pin
   one to zero.
2. So this one is not a contraction at all. Which plain indexing
   expression keeps the batch axis and takes position 0 of the other two?
3. `a[:, 0, 0]` — the drill is here to mark the boundary: repeated-letter
   tricks like `'bii->bi'` read the diagonal, not an arbitrary fixed
   position.

</details>


In [ ]:
import torch as t

def solve(a):
    """Return the 1-D length-b array of each matrix's TOP-LEFT element a[k, 0, 0]."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(293)


In [ ]:
#@title 💡 Solution — Problem 293
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return a[:, 0, 0]


print(solve(t.arange(8.0).reshape(2, 2, 2)))


#### Common mistakes

- **"'ii' is illegal / a typo."** — Within one operand it's the diagonal
  selector. Across operands it's the pairing rule. Same letter, two
  well-defined meanings by placement.
- **"diag(a @ b) requires computing a @ b."** — `'ik,ki->i'` computes just
  the n diagonal dots: O(n²) vs O(n³). When a drill says "without the full
  product", this is what it's fishing for.
- **"'ij,ji->' and 'ij,ij->' are interchangeable."** — Transposed second
  operand: the first is tr(a@b), the second the Frobenius product Σ aᵢⱼbᵢⱼ.
  They agree only for symmetric b. Read letter POSITIONS, not letter SETS.
